In [12]:
import os
from pathlib import Path
import h5py
import numpy as np
import pandas as pd
from tqdm import tqdm
from src.module.utils import load_config, load_npy

In [13]:
config = load_config("../../config/conf.yaml")
data_path = Path(config["data_path"]).expanduser()

# VitalDB (케이스 기준)

In [4]:
vdb_cohort = pd.read_csv(data_path / "vitaldb" / "clinical_data.csv").reset_index(
    drop=True
)
vdb_cohort["age_num"] = np.where(
    (vdb_cohort["age"] == ">89"), 90, vdb_cohort["age"]
).astype(
    np.float32  # age < 1
)
raw_case_no = len(vdb_cohort)
print(f"Case no: {raw_case_no}")

Case no: 6388


In [5]:
vdb_cohort = vdb_cohort.query("18 <= age_num")
step1_case_no = len(vdb_cohort)
print(f"Case no: {step1_case_no}({raw_case_no - step1_case_no} excluded)")

Case no: 6331(57 excluded)


In [8]:
# cardiac surgery는 애초에 없음.
vdb_cohort = vdb_cohort.query("department not in ['Gynecology']")
step2_case_no = len(vdb_cohort)
print(f"Case no: {step2_case_no}({step1_case_no - step2_case_no} excluded)")

Case no: 6103(228 excluded)


In [9]:
vdb_cohort = vdb_cohort.query("ane_type == 'General'")
step3_case_no = len(vdb_cohort)
print(f"Case no: {step3_case_no}({step2_case_no - step3_case_no} excluded)")

Case no: 5788(315 excluded)


In [11]:
# ABP signal without low signal quality
vdb_pos_cid = np.hstack(
    [
        load_npy(fp)
        for fp in (data_path / "vitaldb" / "05.dataset_new").rglob(
            "hypophetversion2*pos_cid.npy"
        )
    ]
)
vdb_neg_cid = np.hstack(
    [
        load_npy(fp)
        for fp in (data_path / "vitaldb" / "05.dataset_new").rglob(
            "hypophetversion2*neg_cid.npy"
        )
    ]
)
vitaldb_caselist = np.unique(np.hstack([vdb_pos_cid, vdb_neg_cid])).astype(int)

In [12]:
vdb_cohort = vdb_cohort.query("caseid.isin(@vitaldb_caselist)").reset_index(drop=True)
step4_case_no = len(vdb_cohort)
print(f"Case no: {step4_case_no}({step3_case_no - step4_case_no} excluded)")
print(f"Patient no: {vdb_cohort.subjectid.nunique()}")

Case no: 3239(2549 excluded)
Patient no: 3118


# MOVER (케이스 기준)

In [24]:
pat_info = pd.read_csv(data_path / "mover" / "patient_information.csv")
pat_code = pd.read_csv(data_path / "mover" / "patient_coding.csv")
pid_to_mrn = pd.read_csv(data_path / "mover" / "EPIC_MRN_PAT_ID.csv")
# Exclude cardiac('02'), gynecologic('10', '0U')
selected_car = pat_code[~pat_code.REF_BILL_CODE.apply(lambda x: str(x)[:2] in ["02"])][
    "MRN"
].unique()
selected_gyn = pat_code[
    ~pat_code.REF_BILL_CODE.apply(lambda x: str(x)[:2] in ["10", "0U"])
]["MRN"].unique()
mov_cohort = (
    pd.merge(pid_to_mrn, pat_info, how="inner", on=["LOG_ID", "MRN"])
    .drop_duplicates()
    .reset_index(drop=True)
)
raw_case_no = len(mov_cohort)
print(f"Case no: {raw_case_no}")

Case no: 64332


In [25]:
mov_cohort = mov_cohort.query("18 <= BIRTH_DATE")
step1_case_no = len(mov_cohort)
print(f"Case no: {step1_case_no}({raw_case_no - step1_case_no} excluded)")

Case no: 64298(34 excluded)


In [26]:
mov_cohort = mov_cohort.query("MRN.isin(@selected_car)")
step2_case_no = len(mov_cohort)
print(f"Case no: {step2_case_no}({step1_case_no - step2_case_no} excluded)")

Case no: 61442(2856 excluded)


In [27]:
# cardiac & obstetric surgery (02, 10, 0U) excluded
mov_cohort = mov_cohort.query("MRN.isin(@selected_gyn)")
step3_case_no = len(mov_cohort)
print(f"Case no: {step3_case_no}({step2_case_no - step3_case_no} excluded)")

Case no: 61440(2 excluded)


In [28]:
mov_cohort = mov_cohort.query("PRIMARY_ANES_TYPE_NM == 'General'")
step4_case_no = len(mov_cohort)
print(f"Case no: {step4_case_no}({step3_case_no - step4_case_no} excluded)")

Case no: 49351(12089 excluded)


In [29]:
# ABP signal without low signal quality
mov_pos_cid = np.hstack(
    [
        load_npy(fp)
        for fp in (data_path / "mover" / "05.dataset_new").rglob(
            "hypophetversion2*pos_cid.npy"
        )
    ]
)
mov_neg_cid = np.hstack(
    [
        load_npy(fp)
        for fp in (data_path / "mover" / "05.dataset_new").rglob(
            "hypophetversion2*neg_cid.npy"
        )
    ]
)
mover_caselist = np.unique(np.hstack([mov_pos_cid, mov_neg_cid]))

In [30]:
mov_cohort = mov_cohort.query("PAT_ID.isin(@mover_caselist)").reset_index(drop=True)
step5_case_no = len(mov_cohort)
print(f"Case no: {step5_case_no}({step4_case_no - step5_case_no} excluded)")
print(f"Patient no: {mov_cohort.PAT_ID.nunique()}")

Case no: 1199(48152 excluded)
Patient no: 649
